HWP **필드(누름틀)** 는 문서 내 특정 위치에 이름표를 붙여두고, 나중에 스크립트로 값을 일괄 주입할 수 있는 기능입니다. 계약서, 납부서, 영수증 등 **같은 양식에 다른 데이터** 를 반복 생성할 때 필수.

hwpapi v0.0.5 부터 Mail Merge 를 위한 **Field API** 가 추가되었습니다.

## Field API 기본

```python
app.create_field(name, memo="", direction="")  # 필드 생성
app.set_field(name, value)                      # 값 주입
app.get_field(name)                             # 값 읽기
app.fields                                      # property: 전체 이름 리스트
app.fields_dict                                 # property: {이름: 값}
app.field_exists(name)                          # 존재 확인
app.move_to_field(name)                         # 필드로 커서 이동
app.delete_field(name)                          # 필드 삭제
app.rename_field(old, new)                      # 이름 변경
app.replace_brackets_with_fields()              # {{name}} → 필드 자동 변환
```

## 사용 패턴 1 — `{{name}}` 브래킷으로 템플릿 작성

가장 간단한 방법. 템플릿 문서에 `{{field_name}}` 형태로 필드 위치를 표시하고, `replace_brackets_with_fields()` 로 한번에 변환합니다.

```python
from hwpapi.core import App

app = App()
app.open("contract_template.hwp")
# 템플릿 내용:
#   계약자: {{name}}
#   계약일: {{date}}
#   계약금액: {{amount}}원

# 브래킷을 실제 필드로 변환
field_names = app.replace_brackets_with_fields()
print(field_names)  # ['name', 'date', 'amount']
```

## 사용 패턴 2 — 한 고객에 대해 필드 채우기

```python
app.set_field("name", "홍길동")
app.set_field("date", "2026-04-15")
app.set_field("amount", "1,200,000")

# 확인
print(app.fields_dict)
# {'name': '홍길동', 'date': '2026-04-15', 'amount': '1,200,000'}

app.save("out/contract_홍길동.hwp")
```

## 사용 패턴 3 — 데이터로부터 100 개 문서 일괄 생성

pandas DataFrame / CSV / list of dict 모두 동일한 흐름.

```python
import pandas as pd
from pathlib import Path
from hwpapi.core import App

# 고객 데이터
customers = pd.read_csv("customers.csv")  # name, date, amount 컬럼

app = App(is_visible=False)   # 대량 처리 시 창 숨기기가 빠름
Path("out").mkdir(exist_ok=True)

# 대화상자 자동 응답
with app.silenced():
    for _, row in customers.iterrows():
        # 템플릿을 매번 새로 열기
        app.open("contract_template.hwp")
        app.replace_brackets_with_fields()   # {{...}} → field

        for col in customers.columns:
            app.set_field(col, str(row[col]))

        app.save(f"out/contract_{row['name']}.hwp")
        app.close()

print(f"✅ {len(customers)}개 문서 생성 완료")
```

## 사용 패턴 4 — 다중 문서 동시 편집

여러 문서를 한 번에 열어놓고 각각에 필드 주입:

```python
app = App()

# 3개 템플릿을 다중 탭으로 열기
docs = [app.documents.open(p) for p in ["tmpl_A.hwp", "tmpl_B.hwp", "tmpl_C.hwp"]]

# 각 문서에서 필드 준비 — xlwings-style 프록시 사용
for doc in docs:
    doc.replace_brackets_with_fields()

# 공통 필드 주입
for doc in docs:
    doc.set_field("company", "(주)예시")
    doc.set_field("ceo", "김대표")
    doc.save()
```

## 대화상자 자동 응답 — `app.silenced()`

자동화 중에 HWP 가 띄우는 확인/경고 다이얼로그를 자동으로 처리합니다. context manager 로 범위 지정.

```python
with app.silenced():                    # 기본: 모든 질문에 YES
    app.open("old.hwp")                  # "변경사항 저장?" → YES
    app.replace_all("2025", "2026")
    app.save("new.hwp")

with app.silenced(mode=0x00200000):     # NO 자동 응답
    app.close()                          # 저장 없이 닫기
```

| 모드값 | 동작 |
|:---|:---|
| `0x00100000` | YES 자동 (기본) |
| `0x00200000` | NO 자동 |
| `0x00300000` | Cancel 자동 |
| `0x00500000` | OK 자동 |
| `0x00000000` | 수동 모드 (사용자 대기) |

context manager 가 종료되면 **자동으로 원래 모드로 복원** 되므로 스크립트 초반 · 중반 · 말미에 따라 다른 정책을 섞어 쓸 수 있습니다.

## pandas 연동 — DataFrame 을 표로, 표를 DataFrame 으로

### DataFrame → HWP 표 삽입

```python
import pandas as pd

df = pd.DataFrame({
    "지역": ["서울", "경기", "인천"],
    "매출": [1850, 1320, 620],
    "전년비": ["+18.5%", "+14.2%", "+8.9%"],
})

app.insert_table(data=df)   # columns 가 자동으로 header 가 됨
```

### HWP 표 → DataFrame

커서를 읽고자 하는 표의 **셀 안** 에 두고 호출:

```python
df = app.read_table()                  # DataFrame
rows = app.read_table(to="list")       # 2D list
csv = app.read_table(to="csv")         # CSV 문자열
```

::: {.callout-note}
HWP 의 표 순회 COM API 는 케이스에 따라 엣지케이스가 있습니다. 표 모양이 규칙적인 일반 표에서 잘 작동하며, 병합된 셀이나 중첩 표가 있는 경우 결과가 예상과 다를 수 있습니다.
:::

## 실전 시나리오 — 급여명세서 100장 생성

```python
import pandas as pd
from pathlib import Path
from hwpapi.core import App

def batch_payroll(csv_path: str, template_path: str, out_dir: str):
    df = pd.read_csv(csv_path)   # name, dept, gross, tax, net 등
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    app = App(is_visible=False)
    with app.silenced():
        for i, row in df.iterrows():
            app.open(template_path)
            app.replace_brackets_with_fields()

            for col in df.columns:
                val = row[col]
                # 숫자는 천단위 구분
                if isinstance(val, (int, float)):
                    val = f"{val:,}"
                app.set_field(col, str(val))

            out = Path(out_dir) / f"payroll_{row['name']}.hwp"
            app.save(str(out))
            app.close()
            print(f"  [{i+1}/{len(df)}] {out.name}")

    app.quit()
    print(f"\n✅ {len(df)}장 생성 완료")

batch_payroll(
    csv_path="employees.csv",
    template_path="payroll_template.hwp",
    out_dir="out/payroll_2026_04",
)
```

## 정리

| 작업 | API |
|:---|:---|
| 템플릿 준비 | `{{field}}` 브래킷 + `replace_brackets_with_fields()` |
| 값 주입 | `app.set_field(name, value)` |
| 값 조회 | `app.get_field(name)` / `app.fields_dict` |
| 자동 응답 | `with app.silenced(): ...` |
| 대량 처리 | `is_visible=False` + 루프 |
| 표 생성 | `app.insert_table(data=df)` |
| 표 읽기 | `app.read_table()` |
| 다중 문서 | `for doc in app.documents: doc.set_field(...)` |

## 참고

- [다중 문서 다루기](08_multi_document.html) — 여러 파일을 한 세션에서 다루기
- [폴더 내 일괄 편집](05_usecase_bulk_edit.html) — 기존 문서 일괄 치환
- [API 레퍼런스](../02_api_reference.html)